# TutorAI with RAG Integration (Week-3 Assignment)

This notebook implements a retrieval-augmented tutoring system based on **Storytelling with Data** by Cole Nussbaumer Knaflic. It uses a rolling window and summary memory to handle long conversations efficiently while maintaining book-grounded answers.


### 1. Import Dependencies

In [ ]:
# Install required libraries if needed
!pip install -q sentence-transformers faiss-cpu huggingface_hub gradio PyPDF2 requests
import os
from typing import List, Tuple, Dict

import faiss
from sentence_transformers import SentenceTransformer
from huggingface_hub import InferenceClient
import gradio as gr
import requests
from io import BytesIO
import PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 16.1 MB/s eta 0:00:00


### 2. Import Inference and embedding models from HuggingFace

In [ ]:
HF_TOKEN = os.getenv("HF_TOKEN")
MODEL_ID = "moonshotai/Kimi-K2-Instruct-0905"
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Initialize client
try:
    client = InferenceClient(model=MODEL_ID, token=HF_TOKEN, provider="together")
except Exception:
    client = None

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### 3. Convert the content of the book into a vector database, and index the summary

In [ ]:
# -----------------------------------------------------------------------------
# Book loading, chunking and indexing
# -----------------------------------------------------------------------------

def load_full_book(pdf_url: str, chunk_words: int = 200) -> Tuple[List[Dict[str, str]], str]:
    """
    Attempt to download and parse the entire PDF book. The book is split into
    chunks of roughly `chunk_words` words and each chunk is tagged with its
    originating page number. A simple heuristic summary is also produced by
    concatenating the first few sentences of early pages. If the download or
    parsing fails, this function returns (None, None).
    """
    try:
        response = requests.get(pdf_url, timeout=60)
        response.raise_for_status()
        pdf_bytes = BytesIO(response.content)
        reader = PyPDF2.PdfReader(pdf_bytes)
    except Exception:
        return None, None

    chunks: List[Dict[str, str]] = []
    summary_parts: List[str] = []
    for page_idx, page in enumerate(reader.pages):
        try:
            text = page.extract_text()
        except Exception:
            text = None
        if not text:
            continue
        words = text.split()
        if len(summary_parts) < 10:
            summary_parts.append(" ".join(words[:50]))
        for i in range(0, len(words), chunk_words):
            chunk_words_list = words[i:i + chunk_words]
            chunk_text = " ".join(chunk_words_list)
            chunks.append({'page': page_idx + 1, 'text': chunk_text})
    summary = " ".join(summary_parts)
    return chunks, summary

PDF_URL = "https://raw.githubusercontent.com/infoalpha/Data-Science-books/master/storytelling-with-data-cole-nussbaumer-knaflic.pdf"
BOOK_CHUNKS, BOOK_SUMMARY = load_full_book(PDF_URL)

book_embeddings = embedder.encode([c['text'] for c in BOOK_CHUNKS], convert_to_numpy=True, normalize_embeddings=True)
embedding_dim = book_embeddings.shape[1]
book_index = faiss.IndexFlatIP(embedding_dim)
book_index.add(book_embeddings)

In [ ]:
print(type(BOOK_CHUNKS), type(BOOK_SUMMARY))
print(BOOK_CHUNKS[:50])
print(len(BOOK_CHUNKS))
# print(BOOK_SUMMARY[:50])
# print(len(BOOK_SUMMARY))

<class 'list'> <class 'str'>
[{'page': 3, 'text': 'storytelling with data'}, {'page': 5, 'text': 'storytelling with data a data visualization guide for business professionals cole nussbaumer knaflic'}, {'page': 6, 'text': 'Cover image: Cole Nussbaumer Knaflic Cover design: Wiley Copyright © 2015 by Cole Nussbaumer Knaflic. All rights reserved. Published by John Wiley & Sons, Inc., Hoboken, New Jersey. Published simultaneously in Canada. No part of this publication may be reproduced, stored in a retrieval system, or transmitted in any form or by any means, electronic, mechanical, photocopying, recording, scanning, or otherwise, except as permitted under Section 107 or 108 of the 1976 United States Copyright Act, without either the prior written permission of the Publisher, or authorization through payment of the appropriate per-copy fee to the Copyright Clearance Center, Inc., 222 Rosewood Drive, Danvers, MA 01923, (978) 750-8400, fax (978) 646-8600, or on the Web at www.copyright.com. 

### 4. Create retrieval and heuristic summary functions to implement RAG

In [ ]:
def retrieve_book(query: str, k: int = 3) -> List[Tuple[str, int]]:
    """Retrieve the top-k most similar chunks from the book index given a query."""
    if not query:
        return []
    q_vec = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    sims, idxs = book_index.search(q_vec, k)
    results: List[Tuple[str, int]] = []
    for idx in idxs[0]:
        chunk = BOOK_CHUNKS[idx]
        results.append((chunk['text'], chunk['page']))
    return results

def make_context(query: str, k: int = 3) -> str:
    ctx_items = retrieve_book(query, k)
    if not ctx_items:
        return ""
    return "".join([f"(p. {page}) {text}" for text, page in ctx_items])

def heuristic_answer(query: str) -> str:
    """Provide a heuristic answer using the precomputed BOOK_SUMMARY as a fallback."""
    return BOOK_SUMMARY

### 5. Chat Generation and History Management

In [ ]:
# -----------------------------------------------------------------------------
# LLM wrapper
# -----------------------------------------------------------------------------
def chat_complete(messages: List[Dict[str, str]], max_tokens: int = 1024, temperature: float = 0.5, top_p: float = 0.9, stream: bool = False) -> str:
    """Wrapper around the HuggingFace InferenceClient for chat completion."""
    if client is None:
        return ""
    try:
        if stream:
            out = []
            for chunk in client.chat_completion(
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature,
                top_p=top_p,
                stream=True,
            ):
                choice = chunk.choices
                delta = getattr(choice.delta, "content", None)
                if delta:
                    out.append(delta)
            return "".join(out)
        resp = client.chat_completion(
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        return resp.choices[0].message.content
    except Exception:
        return ""


# -----------------------------------------------------------------------------
# Conversation management: pruning, summarisation, message construction
# -----------------------------------------------------------------------------
MAX_TURNS_WINDOW = 20
SUMMARIZE_AFTER_TURNS = 19


def make_base_messages(summary_note: str = "") -> List[Dict[str, str]]:
    base = [
        {"role": "system", "content": "You are a helpful tutor who answers questions strictly from the provided book on data storytelling. Always cite page numbers from the book in parentheses. If the answer is not in the book, say you don’t know and do not make up answers."}
    ]
    if summary_note:
        base.append({"role": "system", "content": f"Conversation summary so far: {summary_note}"})
    return base

def prune_history(history: List[Tuple[str, str]], max_pairs: int = MAX_TURNS_WINDOW) -> List[Tuple[str, str]]:
    return history[-max_pairs:]

def to_messages(base_messages: List[Dict[str, str]], history_pairs: List[Tuple[str, str]], user_msg: str) -> List[Dict[str, str]]:
    messages = list(base_messages)
    for role, content in history_pairs:
        messages.append({"role": role, "content": content})
    messages.append({"role": "user", "content": user_msg})
    return messages

def summarize_history_if_needed(history: List[Tuple[str, str]], summary_note: str) -> str:
    if len(history) < SUMMARIZE_AFTER_TURNS:
        return summary_note
    base = make_base_messages("")
    msgs: List[Dict[str, str]] = list(base)
    summarise_prompt = (
        "Summarise the conversation so far into 3-4 bullet points focusing on the user's learning goals and any constraints. "
        "Keep it under 200 words."
    )
    transcript_lines: List[str] = []
    for role, content in history[-60:]:
        prefix = "U" if role == "user" else "A"
        transcript_lines.append(f"{prefix}: {content}")
    transcript = "".join(transcript_lines)
    msgs.append({"role": "user", "content": summarise_prompt + "Transcript:" + transcript})
    summary = chat_complete(msgs, max_tokens=500, temperature=0.2, top_p=0.9)
    if summary:
        return summary.strip()
    return summary_note

### 6. Create a Tutor based on the book

In [ ]:

# -----------------------------------------------------------------------------
# Response generation and Tutor class
# -----------------------------------------------------------------------------

def generate_response(history: List[Tuple[str, str]], summary_note: str, query: str) -> Tuple[str, str]:
    new_summary = summarize_history_if_needed(history, summary_note)
    context = make_context(query, k=3)
    base_messages = make_base_messages(new_summary)
    recent_history = prune_history(history, MAX_TURNS_WINDOW)
    user_msg = f"{query} Context: {context}" if context else query
    messages = to_messages(base_messages, recent_history, user_msg)
    reply = chat_complete(messages, max_tokens=1024, temperature=0.4, top_p=0.95, stream=False)
    if not reply or "(p." not in reply:
        reply = heuristic_answer(query)
    return reply, new_summary

class BookTutor:
    def __init__(self):
        self.history: List[Tuple[str, str]] = []
        self.summary: str = ""

    def ask(self, query: str) -> str:
        reply, new_summary = generate_response(self.history, self.summary, query)
        self.history.append(("user", query))
        self.history.append(("assistant", reply))
        self.summary = new_summary
        return reply


In [ ]:
# -----------------------------------------------------------------------------
# Interactive Gradio interface with summarisation and rolling memory
# -----------------------------------------------------------------------------

tutor = BookTutor()

with gr.Blocks() as app:
    gr.Markdown("""## TutorAI – Storytelling with Data Chat

Ask questions about the book *Storytelling with Data*. The tutor answers strictly from the text, includes citations (page numbers), and summarises long conversations to keep performance high.
""")
    chatbot = gr.Chatbot(height=400)
    msg_in = gr.Textbox(label="Your question", placeholder="Ask about chart types, design principles, storytelling techniques, ...", lines=1)
    clear_btn = gr.Button("Clear chat")

    def respond(user_message, chat_history):
        if chat_history is None:
            chat_history = []
        reply = tutor.ask(user_message)
        chat_history = chat_history + [[user_message, reply]]
        return "", chat_history

    def clear(chat_history):
        tutor.history = []
        tutor.summary = ""
        return [], []

    msg_in.submit(respond, [msg_in, chatbot], [msg_in, chatbot])
    clear_btn.click(clear, chatbot, chatbot)

# Launch the app in a new browser tab
app.launch(inbrowser=True)


/tmp/ipython-input-1936041116.py:12: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://797c509ad18da020b9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
